# GINO — Geometry-Informed Neural Operator

Trening, vizuelizacija i evaluacija GINO arhitekture na lid-driven cavity problemu.

Pipeline: **MeshGNN → GNO enkoder → FNO jezgro → GNO dekoder → MLP glava**.
Koristi isti `NavierStokesLoss` kao `main.ipynb`, ali batch je **jedno `(re, time)` stanje** (operator enkodira cijelo polje odjednom).

In [ ]:
%cd ../..

In [1]:
import pathlib

import yaml
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import torch
from torch import nn

from IPython.display import HTML, display
from pprint import pprint

import src.visuals as visual
import src.utils as utils

from src.models_gino import GINO
from src.loss import NavierStokesLoss
from src.dataloader import load_data, gen_state_dataloaders
from src.train import train_model

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)


frac_size = 0.1
file_name = "data"
data_path = pathlib.Path(f"data/real_path/frac_{int(100*frac_size)}")

train_df, valid_df, test_df = load_data(data_path, file_name)

input_col_names = ['time', 're', 'x', 'y']
target_col_names = ['U_x', 'U_y', 'p']

# Arhitektura + hiperparametri iz config-a
cfg = yaml.safe_load(open("config/architecture/gino.yaml"))
group_cols = cfg['training']['group_cols']

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [3]:
train_df.describe()

,time,re,x,y,U_x,U_y,p
count,180224.000000,180224.000000,180224.000000,180224.000000,180224.000000,180224.000000,180224.000000
mean,0.250000,730.000000,0.044181,0.050000,0.000072,0.015573,-0.013369
std,0.158114,201.246676,0.028746,0.028864,0.191900,0.119423,0.105119
min,0.000000,460.000000,0.000541,0.000781,-0.238965,-0.523099,-3.660870
25%,0.100000,595.000000,0.019156,0.025391,-0.083716,-0.009605,-0.038047
50%,0.250000,730.000000,0.041292,0.050000,-0.028560,0.004208,-0.001017
75%,0.400000,865.000000,0.067615,0.074609,-0.000230,0.071870,0.004474
max,0.500000,1000.000000,0.098918,0.099219,0.953934,0.330282,3.227460


In [4]:
valid_df.describe()

,time,re,x,y,U_x,U_y,p
count,45056.000000,45056.0,45056.000000,45056.000000,45056.000000,45056.000000,45056.000000
mean,0.250000,100.0,0.044181,0.050000,0.000074,0.018041,-0.036866
std,0.158116,0.0,0.028746,0.028864,0.203814,0.132093,0.657614
min,0.000000,100.0,0.000541,0.000781,-0.208126,-0.391485,-17.771900
25%,0.100000,100.0,0.019156,0.025391,-0.099851,-0.023018,-0.077190
50%,0.250000,100.0,0.041292,0.050000,-0.033492,0.001291,0.000000
75%,0.400000,100.0,0.067615,0.074609,-0.000120,0.075248,0.066330
max,0.500000,100.0,0.098918,0.099219,0.954127,0.363694,13.309000


In [5]:
test_df.describe()

,time,re,x,y,U_x,U_y,p
count,180224.000000,180224.0,180224.000000,180224.000000,180224.000000,180224.000000,180224.000000
mean,0.250000,280.0,0.044225,0.050000,0.000019,0.017213,-0.020077
std,0.158114,0.0,0.028750,0.028867,0.201809,0.128570,0.261113
min,0.000000,280.0,0.000271,0.000391,-0.211943,-0.428904,-12.666400
25%,0.100000,280.0,0.019038,0.025195,-0.095638,-0.017849,-0.051836
50%,0.250000,280.0,0.041356,0.050000,-0.032471,0.002104,0.000000
75%,0.400000,280.0,0.067897,0.074805,-0.000133,0.074854,0.019197
max,0.500000,280.0,0.099459,0.099609,0.977064,0.357944,9.829750


In [6]:
mean = train_df.mean()
std = train_df.std()

train_df = utils.normalize_data(train_df, mean, std)
valid_df = utils.normalize_data(valid_df, mean, std)
test_df = utils.normalize_data(test_df, mean, std)

In [7]:
# Per-state loaderi: svaki batch je jedno (re, time) stanje (cijelo polje).
train_dataloader, valid_dataloader, test_dataloader = gen_state_dataloaders(
    train_df,
    valid_df,
    test_df,
    input_col_names,
    target_col_names,
    group_cols=group_cols,
)

print(f"Trening stanja: {len(train_dataloader)}")
print(f"Validaciona stanja: {len(valid_dataloader)}")
print(f"Test stanja: {len(test_dataloader)}")

Trening stanja: 44
Validaciona stanja: 11
Test stanja: 11


In [8]:
model = GINO.from_config(cfg).to(device)
criterion = NavierStokesLoss(cfg['training']['physics_weight'], mean, std)
optimizer = torch.optim.Adam(model.parameters())

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=15
)

EPOCHS = 500

In [9]:
n_param = sum([p.numel() for p in model.parameters()])

print("Number of parameters: ", n_param)

Number of parameters:  1196995


## Trening

> Napomena: jedan korak = jedno stanje (cijelo polje). Trening je znatno brzi na GPU-u; na CPU-u je `torch.cdist` k-NN usko grlo. Smanji `EPOCHS` ili `latent_size` u config-u za brzu probu.

In [10]:
run_dir = utils.create_run_directory()

history = train_model(
    model,
    train_dataloader,
    valid_dataloader,
    criterion,
    optimizer,
    scheduler,
    device,
    EPOCHS,
    run_dir,
    checkpoint=None,
    physics_loss=True,
)

100%|██████████| 44/44 [00:47<00:00,  1.08s/it]


Epoch 0: train=0.907146 (data=0.906939, physics=0.002078) | valid=12.684340 (data=12.684076, physics=0.002640)


100%|██████████| 44/44 [00:57<00:00,  1.30s/it]


Epoch 1: train=0.450115 (data=0.449728, physics=0.003876) | valid=11.946015 (data=11.945554, physics=0.004605)


 27%|██▋       | 12/44 [00:17<00:46,  1.46s/it]


KeyboardInterrupt: 

In [ ]:
def plot(df, ax):
    for x in ax:
        x.ticklabel_format(axis='y', style='sci', scilimits=(-2,2), useMathText=True)

    df[["train", "valid"]].plot(ax=ax[0])
    df[["train_data", "valid_data"]].plot(ax=ax[1])
    df[["train_physics", "valid_physics"]].plot(ax=ax[2])

history_df = pd.DataFrame(history)

fig, ax = plt.subplots(4, 3, figsize=(12, 12), tight_layout=True)

plot(history_df, ax[0])
plot(history_df[:100], ax[1])
plot(history_df[100:300], ax[2])
plot(history_df[300:], ax[3])

fig.savefig(run_dir / "losses.png")
history_df.to_csv(run_dir / "losses.csv")

In [ ]:
# Ucitaj originalne (nenormalizovane) podatke za vizuelizaciju i evaluaciju
train_df_original, valid_df_original, test_df_original = load_data(data_path, file_name)

train_df_original = train_df_original.sort_values(["re", "time", "y", "x"]).reset_index(drop=True)
valid_df_original = valid_df_original.sort_values(["re", "time", "y", "x"]).reset_index(drop=True)
test_df_original  = test_df_original.sort_values(["re", "time", "y", "x"]).reset_index(drop=True)

print(f"Train skup: {train_df_original.shape[0]} redova")
print(f"Valid skup: {valid_df_original.shape[0]} redova")
print(f"Test skup: {test_df_original.shape[0]} redova")
print(f"\nVremenski koraci: {sorted(train_df_original['time'].unique().tolist())}")
print(f"Reynolds brojevi: {sorted(train_df_original['re'].unique().tolist())}")

## Vizuelizacija

In [ ]:
re_values = sorted(test_df_original['re'].unique().tolist())
time_steps = sorted(test_df_original['time'].unique().tolist())

print(f"Dostupni Reynolds brojevi: {re_values[:5]}... (ukupno {len(re_values)})")
print(f"Dostupni vremenski koraci: {time_steps}")

# Ground-truth stanje (samo podaci)
visual.plot_velocity_and_pressure(test_df_original, time_steps[5], re_values[5], "Test:")

In [ ]:
# Evolucija toka kroz vrijeme (samo podaci)
visual.plot_evolution_in_time(test_df_original, re_values[5])

In [ ]:
# Poredjenje GINO predikcije sa stvarnim podacima (radi po-stanju -> kompatibilno sa GINO)
visual.compare_predictions(model, test_df_original, time_steps[3], re_values[5], mean, std, device)

In [ ]:
# Animacija: Ground Truth vs GINO predikcija kroz vrijeme za jedan Re
visual.animate_truth_vs_pred(model, test_df_original, re_values[5],
                             mean, std, device, fps=2,
                             output_file=run_dir / "animations" / "truth_vs_pred.gif")

## Evaluacija

In [ ]:
def evaluate_gino(model, df, input_cols, target_cols, mean, std, device, group_cols=('re', 'time')):
    """Per-state evaluacija za operator: model se poziva na svakom (re, time) stanju posebno,
    pa se metrike racunaju na originalnoj skali (kao utils.evaluate_model, ali bez mijesanja stanja)."""
    model.eval()
    im, istd = mean[input_cols].to_numpy(np.float32), std[input_cols].to_numpy(np.float32)
    tm, tstd = mean[target_cols].to_numpy(np.float32), std[target_cols].to_numpy(np.float32)

    preds, trues = [], []
    for _, g in df.groupby(list(group_cols), sort=False):
        x = (g[input_cols].to_numpy(np.float32) - im) / istd
        xt = torch.tensor(x, dtype=torch.float32, device=device)
        with torch.no_grad():
            p = model(xt).cpu().numpy()
        preds.append(p * tstd + tm)
        trues.append(g[target_cols].to_numpy(np.float32))

    y_pred, y_true = np.concatenate(preds), np.concatenate(trues)
    metrics = {"all": utils.compute_metrics(y_true, y_pred)}
    for i, c in enumerate(target_cols):
        metrics[c] = utils.compute_metrics(y_true[:, i], y_pred[:, i])
    return metrics

In [ ]:
# Puni evaluacioni pregled po splitovima
split_dfs = {
    "valid": valid_df_original,
    "test": test_df_original,
}

for split_name, split_df in split_dfs.items():
    print(f"\n{'=' * 60}")
    print(f"{split_name.upper()} EVALUATION")
    print(f"{'=' * 60}")

    data_metrics = evaluate_gino(
        model, split_df, input_col_names, target_col_names,
        mean, std, device, group_cols=group_cols,
    )

    print(f"\nSamples: {len(split_df)}")
    print("\nSupervised metrics (original scale):")
    pprint(data_metrics)